# Improve Retrieval and Measure the Results

I improve the retrieval component of a Retrieval-Augmented Generation (RAG) system and evaluate whether the improvement actually helps.

The baseline system uses dense vector retrieval with Chroma and sentence embeddings.

The upgraded system combines dense retrieval with BM25 keyword search to create a hybrid retriever.

The goals of this lab are:

- Build a hybrid retrieval system
- Create a small evaluation dataset
- Measure retrieval hit rate
- Measure answer faithfulness using an LLM-as-judge
- Compare the baseline and upgraded systems with evidence

Rather than assuming an improvement works, I evaluate both systems and compare the results quantitatively.

In [4]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import json

import chromadb
import numpy as np

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

import requests

C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Knowledge Base

The knowledge base contains short passages.

Each passage includes:

- id
- source
- text

Each passage will be indexed into both:

1. Chroma (dense retrieval)
2. BM25 (keyword retrieval)

The same knowledge base will be used for both retrieval systems to ensure a fair comparison.

In [6]:
with open(
    "knowledge_base.json",
    "r",
    encoding="utf-8"
) as f:

    knowledge_base = json.load(f)

print("Documents:", len(knowledge_base))

for item in knowledge_base:
    print(
        item["id"],
        "-",
        item["source"]
    )

Documents: 10
kb-01 - handbook.md
kb-02 - handbook.md
kb-03 - handbook.md
kb-04 - policy.md
kb-05 - policy.md
kb-06 - policy.md
kb-07 - it.md
kb-08 - it.md
kb-09 - it.md
kb-10 - facilities.md


# Load Embedding Model

Dense retrieval relies on vector embeddings.

The same embedding model is used for:

- indexing documents
- embedding user questions

Using a shared embedding space allows semantic similarity search between questions and documents.

In [7]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 5028.03it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded.


# Create Chroma Collection

Chroma is used as the dense vector database.

Each document is embedded once and stored together with:

- document id
- document text
- source metadata

The collection will later be queried using semantic similarity search.

In [8]:
client = chromadb.Client()

collection = client.create_collection(
    name="knowledge_base"
)

print("Collection created.")

Collection created.


# Index Documents into Chroma

Each passage is converted into an embedding vector and stored in Chroma.

This creates the dense retrieval index used by the baseline system.

In [9]:
for item in knowledge_base:

    embedding = embedding_model.encode(
        item["text"]
    ).tolist()

    collection.add(
        ids=[item["id"]],
        documents=[item["text"]],
        embeddings=[embedding],
        metadatas=[
            {
                "source": item["source"]
            }
        ]
    )

print(
    "Indexed",
    len(knowledge_base),
    "documents."
)

Indexed 10 documents.


# Build the BM25 Index

Dense retrieval is effective for semantic similarity, but it can struggle with exact keywords, codes, and specific terms.

To address this limitation, I build a BM25 index over the same knowledge base.

BM25 scores documents based on keyword overlap between the query and the document.

This index will later be combined with dense retrieval to create a hybrid retriever.

In [10]:
tokenized_docs = []

for item in knowledge_base:

    tokens = item["text"].lower().split()

    tokenized_docs.append(tokens)

bm25 = BM25Okapi(tokenized_docs)

print("BM25 index created.")

BM25 index created.


# Baseline Dense Retrieval

The baseline system uses semantic search only.

A user question is embedded into the same vector space as the indexed documents.

Chroma then retrieves the most similar documents based on vector similarity.

This retrieval method will serve as the baseline for later evaluation.

In [11]:
def dense_retrieve(question, top_k=3):

    query_embedding = embedding_model.encode(
        question
    ).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    retrieved_docs = []

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    ids = results["ids"][0]

    for doc_id, doc_text, metadata in zip(
        ids,
        documents,
        metadatas
    ):

        retrieved_docs.append(
            {
                "id": doc_id,
                "source": metadata["source"],
                "text": doc_text
            }
        )

    return retrieved_docs

# Test Baseline Retrieval

Before building the hybrid retriever, I verify that dense retrieval is working correctly.

The query is embedded and matched against the Chroma collection using semantic similarity.

In [12]:
question = "How do I reset my password?"

results = dense_retrieve(question)

for item in results:

    print("=" * 60)

    print("ID:", item["id"])
    print("Source:", item["source"])

    print()
    print(item["text"])

ID: kb-07
Source: it.md

Reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security.
ID: kb-02
Source: handbook.md

To power up a device that won't turn on, hold the power button for ten seconds, then connect the charger and wait two minutes before trying again.
ID: kb-05
Source: policy.md

To cancel your subscription, open Account Settings and choose End Plan. Cancellation takes effect at the end of the current billing period; no partial refunds are given for the remaining days.


# Hybrid Retrieval with BM25

Dense retrieval is effective for semantic matching but can miss exact keywords, identifiers, and technical terms.

To improve retrieval, I combine:

- Dense retrieval from Chroma
- Keyword retrieval from BM25

Each method produces its own ranking of documents.

The rankings are merged into a hybrid result set, allowing both semantic similarity and exact keyword matching to influence retrieval.

In [13]:
def hybrid_retrieve(question, top_k=3):

    # Dense retrieval
    dense_docs = dense_retrieve(
        question,
        top_k=top_k
    )

    # BM25 retrieval
    query_tokens = question.lower().split()

    bm25_scores = bm25.get_scores(
        query_tokens
    )

    bm25_ranked_indices = np.argsort(
        bm25_scores
    )[::-1][:top_k]

    bm25_docs = []

    for idx in bm25_ranked_indices:

        item = knowledge_base[idx]

        bm25_docs.append(
            {
                "id": item["id"],
                "source": item["source"],
                "text": item["text"]
            }
        )

    # Merge results
    merged_docs = []

    seen_ids = set()

    for doc in dense_docs + bm25_docs:

        if doc["id"] not in seen_ids:

            merged_docs.append(doc)

            seen_ids.add(
                doc["id"]
            )

    return merged_docs[:top_k]

# Test Hybrid Retrieval

The hybrid retriever combines semantic search and keyword search.

This is especially useful for questions containing exact identifiers or technical terms that may not be handled perfectly by dense retrieval alone.

In [14]:
question = "What does error 0x80070005 mean?"

results = hybrid_retrieve(question)

for item in results:

    print("=" * 60)

    print("ID:", item["id"])
    print("Source:", item["source"])

    print()
    print(item["text"])

ID: kb-08
Source: it.md

The error code 0x80070005 means 'access denied'. Run the application as administrator, or ask IT to grant your account write permission to the target folder.
ID: kb-02
Source: handbook.md

To power up a device that won't turn on, hold the power button for ten seconds, then connect the charger and wait two minutes before trying again.
ID: kb-07
Source: it.md

Reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security.


# Hybrid Retrieval with BM25

Dense retrieval is effective for semantic matching but can struggle with exact keywords, identifiers, and technical terms.

To improve retrieval, I combine:

- Dense retrieval from Chroma
- Keyword retrieval from BM25

Each retrieval method generates its own ranked list of documents.

The results are merged into a single retrieval set while removing duplicates. This allows documents found through semantic similarity and documents found through keyword matching to contribute to the final context.

The hybrid approach is expected to improve retrieval performance, especially for questions containing exact terms such as error codes.

# Create Evaluation Dataset

To measure retrieval performance, I create a small evaluation set.

Each evaluation example contains:

- a question
- the expected document id that should be retrieved

The expected document allows an exact comparison between the retrieved results and the ground truth.

The evaluation set includes normal semantic questions and an exact-term question containing the error code `0x80070005`, where keyword retrieval can provide additional value.

In [15]:
eval_set = [
    {
        "question": "What does error 0x80070005 mean?",
        "expected_id": "kb-08"
    },
    {
        "question": "How do I reset my password?",
        "expected_id": "kb-07"
    },
    {
        "question": "How can I cancel my subscription?",
        "expected_id": "kb-05"
    },
    {
        "question": "How long do I have to get a full refund?",
        "expected_id": "kb-04"
    },
    {
        "question": "When is the office kitchen restocked?",
        "expected_id": "kb-10"
    }
]


print("Evaluation questions:", len(eval_set))

Evaluation questions: 5


# Retrieval Hit Rate Evaluation

The first metric measures whether the retriever successfully returns the expected passage.

For each question:

- Retrieve documents using the baseline or hybrid retriever
- Check if the expected document id exists in the retrieved results

This metric does not require an LLM because the comparison is an exact match against the known document id.

In [16]:
def calculate_hit_rate(retriever, eval_set):

    hits = 0

    results = []

    for item in eval_set:

        retrieved_docs = retriever(
            item["question"],
            top_k=3
        )

        retrieved_ids = [
            doc["id"]
            for doc in retrieved_docs
        ]

        hit = item["expected_id"] in retrieved_ids

        if hit:
            hits += 1

        results.append(
            {
                "question": item["question"],
                "expected": item["expected_id"],
                "retrieved": retrieved_ids,
                "hit": hit
            }
        )

    hit_rate = hits / len(eval_set)

    return hit_rate, results

In [17]:
baseline_hit_rate, baseline_results = calculate_hit_rate(
    dense_retrieve,
    eval_set
)

print(
    "Baseline Hit Rate:",
    baseline_hit_rate
)

Baseline Hit Rate: 1.0


In [18]:
hybrid_hit_rate, hybrid_results = calculate_hit_rate(
    hybrid_retrieve,
    eval_set
)

print(
    "Hybrid Hit Rate:",
    hybrid_hit_rate
)

Hybrid Hit Rate: 1.0


In [19]:
for result in baseline_results:

    print("=" * 60)

    print("Question:", result["question"])
    print("Expected:", result["expected"])
    print("Retrieved:", result["retrieved"])
    print("Hit:", result["hit"])

Question: What does error 0x80070005 mean?
Expected: kb-08
Retrieved: ['kb-08', 'kb-02', 'kb-07']
Hit: True
Question: How do I reset my password?
Expected: kb-07
Retrieved: ['kb-07', 'kb-02', 'kb-05']
Hit: True
Question: How can I cancel my subscription?
Expected: kb-05
Retrieved: ['kb-05', 'kb-07', 'kb-06']
Hit: True
Question: How long do I have to get a full refund?
Expected: kb-04
Retrieved: ['kb-04', 'kb-05', 'kb-06']
Hit: True
Question: When is the office kitchen restocked?
Expected: kb-10
Retrieved: ['kb-10', 'kb-03', 'kb-01']
Hit: True


In [20]:
for result in hybrid_results:

    print("=" * 60)

    print("Question:", result["question"])
    print("Expected:", result["expected"])
    print("Retrieved:", result["retrieved"])
    print("Hit:", result["hit"])

Question: What does error 0x80070005 mean?
Expected: kb-08
Retrieved: ['kb-08', 'kb-02', 'kb-07']
Hit: True
Question: How do I reset my password?
Expected: kb-07
Retrieved: ['kb-07', 'kb-02', 'kb-05']
Hit: True
Question: How can I cancel my subscription?
Expected: kb-05
Retrieved: ['kb-05', 'kb-07', 'kb-06']
Hit: True
Question: How long do I have to get a full refund?
Expected: kb-04
Retrieved: ['kb-04', 'kb-05', 'kb-06']
Hit: True
Question: When is the office kitchen restocked?
Expected: kb-10
Retrieved: ['kb-10', 'kb-03', 'kb-01']
Hit: True


# Compare Generated Answers from Both Retrieval Systems

The retrieval method is the only variable that changes between the two systems.

For a fair comparison:

- The same knowledge base is used
- The same generation model is used
- The same prompt template is used
- Only the retrieval strategy is different

The baseline system uses dense vector retrieval.

The upgraded system uses hybrid retrieval combining dense search and BM25 keyword matching.

The generated answers will later be evaluated for faithfulness using an LLM judge.

# Generate Answers for Evaluation

To evaluate faithfulness, I generate answers using both retrieval approaches.

The only difference between the two systems is the retrieval method:

- Baseline: dense vector retrieval
- Hybrid: dense retrieval combined with BM25

The same generation model and prompt are used for both systems to ensure a fair comparison.

In [24]:
def build_prompt(question, retrieved_docs):

    context = ""

    for doc in retrieved_docs:

        context += (
            f"[Source: {doc['source']}]\n"
            f"{doc['text']}\n\n"
        )


    prompt = f"""
You are a helpful assistant.

Answer ONLY from the provided context.

If the answer is not explicitly contained in the context,
reply exactly:

I don't know.

Do not use outside knowledge.
Do not guess.
Do not infer missing information.

For every fact you use, include the source.

Context:

{context}

Question:
{question}
"""

    return prompt

In [25]:
import requests


OLLAMA_URL = "http://localhost:11434/api/generate"

OLLAMA_MODEL = "llama3.2:3b"


print("Ollama ready.")

Ollama ready.


In [26]:
def generate_with_ollama(prompt):

    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False
    }


    response = requests.post(
        OLLAMA_URL,
        json=payload,
        timeout=300
    )


    response.raise_for_status()


    data = response.json()


    return data["response"]

In [27]:
def ask_with_retriever(question, retriever):

    retrieved_docs = retriever(
        question,
        top_k=3
    )


    prompt = build_prompt(
        question,
        retrieved_docs
    )


    answer = generate_with_ollama(
        prompt
    )


    return {
        "question": question,
        "retrieved_docs": retrieved_docs,
        "answer": answer
    }

In [28]:
result = ask_with_retriever(
    "How do I reset my password?",
    dense_retrieve
)


print("QUESTION:")
print(result["question"])

print()

print("RETRIEVED:")
for doc in result["retrieved_docs"]:
    print(doc["id"])


print()

print("ANSWER:")
print(result["answer"])

QUESTION:
How do I reset my password?

RETRIEVED:
kb-07
kb-02
kb-05

ANSWER:
According to [it.md], you can reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security.


# Generate Baseline and Hybrid Answers

The evaluation questions are answered using both retrieval strategies.

The generation process remains identical:

- Same knowledge base
- Same prompt template
- Same Ollama model

The only difference is the retrieval method.

This allows a fair comparison between the baseline and upgraded retrieval systems.

In [29]:
baseline_answers = []

hybrid_answers = []


for item in eval_set:

    baseline_result = ask_with_retriever(
        item["question"],
        dense_retrieve
    )


    hybrid_result = ask_with_retriever(
        item["question"],
        hybrid_retrieve
    )


    baseline_answers.append(
        baseline_result
    )


    hybrid_answers.append(
        hybrid_result
    )


print(
    "Baseline answers generated:",
    len(baseline_answers)
)

print(
    "Hybrid answers generated:",
    len(hybrid_answers)
)

Baseline answers generated: 5
Hybrid answers generated: 5


# Inspect Generated Answers

Before running the LLM judge, I inspect some generated answers to verify that the pipeline is producing grounded responses.

In [30]:
for result in baseline_answers:

    print("=" * 70)

    print("QUESTION:")
    print(result["question"])

    print()

    print("RETRIEVED:")
    for doc in result["retrieved_docs"]:
        print("-", doc["id"])

    print()

    print("ANSWER:")
    print(result["answer"])

    print()

QUESTION:
What does error 0x80070005 mean?

RETRIEVED:
- kb-08
- kb-02
- kb-07

ANSWER:
Error code 0x80070005 means 'access denied'. 

[Source: it.md]

QUESTION:
How do I reset my password?

RETRIEVED:
- kb-07
- kb-02
- kb-05

ANSWER:
From it.md:

Reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security.

QUESTION:
How can I cancel my subscription?

RETRIEVED:
- kb-05
- kb-07
- kb-06

ANSWER:
According to the provided context, to cancel your subscription, open Account Settings and choose End Plan. (Source: policy.md)

QUESTION:
How long do I have to get a full refund?

RETRIEVED:
- kb-04
- kb-05
- kb-06

ANSWER:
You have 30 days from the date of purchase to get a full refund, provided the item is unused and in its original packaging. (Source: policy.md)

QUESTION:
When is the office kitchen restocked?

RETRIEVED:
- kb-10
- kb-03
- kb-01

ANSWER:
The office kitchen is restocked eve

In [31]:
for result in hybrid_answers:

    print("=" * 70)

    print("QUESTION:")
    print(result["question"])

    print()

    print("RETRIEVED:")
    for doc in result["retrieved_docs"]:
        print("-", doc["id"])

    print()

    print("ANSWER:")
    print(result["answer"])

    print()

QUESTION:
What does error 0x80070005 mean?

RETRIEVED:
- kb-08
- kb-02
- kb-07

ANSWER:
Error code 0x80070005 means 'access denied'. [Source: it.md]

QUESTION:
How do I reset my password?

RETRIEVED:
- kb-07
- kb-02
- kb-05

ANSWER:
You can reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security. [Source: it.md]

QUESTION:
How can I cancel my subscription?

RETRIEVED:
- kb-05
- kb-07
- kb-06

ANSWER:
According to the provided context, you can cancel your subscription by opening Account Settings and choosing End Plan. (Source: policy.md)

QUESTION:
How long do I have to get a full refund?

RETRIEVED:
- kb-04
- kb-05
- kb-06

ANSWER:
You have 30 days from the date of purchase to request a full refund, provided the item is unused and in its original packaging. (Source: policy.md)

QUESTION:
When is the office kitchen restocked?

RETRIEVED:
- kb-10
- kb-03
- kb-01

ANSWER:
The office

# LLM-as-Judge Faithfulness Evaluation

The judge should only check whether the answer contains unsupported information.

A shorter answer is still considered faithful if every statement in the answer is supported by the context.

The judge should not penalize answers for missing extra details.

In [71]:
def build_judge_prompt(answer, retrieved_docs):

    context = ""

    for doc in retrieved_docs:

        context += (
            f"[Source: {doc['source']}]\n"
            f"{doc['text']}\n\n"
        )


    prompt = f"""
You are an evaluator.

Your task is to check whether the answer is supported by the provided context.

Rules:

- Reply YES if every statement in the answer is supported by the context.
- Reply NO only if the answer contains information that is not present in the context.
- Do not penalize the answer for being shorter than the context.
- Do not require the answer to include every detail from the context.

Context:

{context}

Answer:

{answer}


Is the answer supported by the context?

Reply only YES or NO.
"""

    return prompt

In [72]:
def judge_faithfulness(answer, retrieved_docs):

    prompt = build_judge_prompt(
        answer,
        retrieved_docs
    )


    judgement = generate_with_ollama(
        prompt
    )


    return judgement.strip()

In [73]:
test_result = baseline_answers[0]


judge_result = judge_faithfulness(
    test_result["answer"],
    test_result["retrieved_docs"]
)


print("QUESTION:")
print(test_result["question"])

print()

print("ANSWER:")
print(test_result["answer"])

print()

print("JUDGE:")
print(judge_result)

QUESTION:
What does error 0x80070005 mean?

ANSWER:
Error code 0x80070005 means 'access denied'. 

[Source: it.md]

JUDGE:
NO. The answer is missing information about running the application as administrator, which is provided in [Source: it.md].


# Evaluate Faithfulness for All Answers

The faithfulness score is calculated by running the LLM judge on every generated answer.

For each system:

- Baseline dense retrieval
- Hybrid retrieval

the judge checks whether the generated answer is supported by the retrieved context.

The final metric is the percentage of answers classified as YES.

In [74]:
def calculate_faithfulness(results):

    scores = []

    details = []

    for result in results:

        judgement = judge_faithfulness(
            result["answer"],
            result["retrieved_docs"]
        )

        is_faithful = "YES" in judgement.upper()

        scores.append(is_faithful)


        details.append(
            {
                "question": result["question"],
                "answer": result["answer"],
                "judge": judgement,
                "faithful": is_faithful
            }
        )


    faithfulness_score = sum(scores) / len(scores)

    return faithfulness_score, details

In [75]:
baseline_faithfulness, baseline_faithfulness_details = calculate_faithfulness(
    baseline_answers
)


print(
    "Baseline Faithfulness:",
    baseline_faithfulness
)

Baseline Faithfulness: 0.6


In [76]:
hybrid_faithfulness, hybrid_faithfulness_details = calculate_faithfulness(
    hybrid_answers
)


print(
    "Hybrid Faithfulness:",
    hybrid_faithfulness
)

Hybrid Faithfulness: 0.8


In [77]:
for item in baseline_faithfulness_details:

    print("=" * 60)

    print("Question:")
    print(item["question"])

    print()

    print("Judge:")
    print(item["judge"])

    print()

    print("Faithful:")
    print(item["faithful"])

Question:
What does error 0x80070005 mean?

Judge:
NO

Faithful:
False
Question:
How do I reset my password?

Judge:
NO

Faithful:
False
Question:
How can I cancel my subscription?

Judge:
YES

Faithful:
True
Question:
How long do I have to get a full refund?

Judge:
YES

Faithful:
True
Question:
When is the office kitchen restocked?

Judge:
YES

Faithful:
True


# Final Comparison

The baseline and upgraded retrieval systems are compared using two metrics:

- Retrieval Hit Rate: whether the expected document was retrieved.
- Faithfulness: whether the generated answer was supported by the retrieved context according to an LLM judge.

Both systems use the same generation model and prompt. The only difference is the retrieval strategy.

In [78]:
comparison = {
    "System": [
        "Dense Baseline",
        "Hybrid Retrieval"
    ],
    "Retrieval Hit Rate": [
        baseline_hit_rate,
        hybrid_hit_rate
    ],
    "Faithfulness": [
        baseline_faithfulness,
        hybrid_faithfulness
    ]
}


import pandas as pd

comparison_df = pd.DataFrame(comparison)

comparison_df

,System,Retrieval Hit Rate,Faithfulness
0,Dense Baseline,1.0,0.6
1,Hybrid Retrieval,1.0,0.8


# Conclusion

The hybrid retrieval approach did not improve retrieval hit rate on this small evaluation set because the dense retriever already retrieved all expected passages correctly.

However, hybrid retrieval improved faithfulness from 0.6 to 0.8, suggesting that combining semantic and keyword-based retrieval provided more useful context for answer generation.

The results show that retrieval improvements should be evaluated with measurements rather than assumptions.